In [7]:
import requests
import time
import markdownify

In [4]:
from mesa.constants import PUBLIC_DIR
lesson_dir = PUBLIC_DIR / 'canvas' / 'ngram_model'

ERROR: You must set the GITLAB_PRIVATE_TOKEN in /home/hobs/code/mesa_python/mesa_python/.env e.g.: `GITLAB_PRIVATE_TOKEN=12345`


PRIVATE_DIR = "/home/hobs/Dropbox/notes/obsidian20250923/2026/mesa-feb26"


In [5]:
titles = [
    'Python_(programming_language)',
    'Python_syntax_and_semantics',
    'History_of_Python',
    'Expression_(computer_science)',
    'Statement_(computer_science)',
    'Computer_science',
    ]
wikipedia = 'https://en.wikipedia.org/wiki'

In [8]:
html_texts, md_texts = [], []
for title in titles:
    path = (lesson_dir / title).with_suffix('.html')
    try:
        with open(path) as fin:
            html_texts.append(fin.read())
    except Exception:
        time.sleep(3)
        print(f'Downloading {title}...')
        html_texts.append(requests.get(
           f'{wikipedia}/{title}',
           headers=headers,
           ).text
        )
        with open(path, 'wt') as fout:
            fout.write(html_texts[-1])
    print(title, html_texts[-1][:80])
    md_texts.append(markdownify.markdownify(html_texts[-1]))
    print(f'"{title}" contained {len(md_texts[-1)} natural language (markdown) characters.')
    with open(path.with_suffix('.md'), 'wt') as fout:
        fout.write(md_texts[-1])

Python_(programming_language) <!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabl
Python_(programming_language) Python (programming language) - Wikipedia

[Jump to content](#bodyContent)

Main menu

Main menu

move to sidebar
hide

Navigation

* [Main page](/wiki/Main_Pag
Python_syntax_and_semantics <!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabl
Python_syntax_and_semantics Python syntax and semantics - Wikipedia

[Jump to content](#bodyContent)

Main menu

Main menu

move to sidebar
hide

Navigation

* [Main page](/wiki/Main_Page 
History_of_Python <!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabl
History_of_Python History of Python - Wikipedia

[Jump to content](#bodyContent)

Main menu

Main menu

move to sidebar
hide

Navigation

* [Main page](/wiki/Main_Page "Visit the
Expression_(computer_science) <!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabl
Expression_

In [12]:
text = '\n'.join(md_texts)
tokens = list(text)
len(tokens)

589709

In [16]:
def collect_ngrams(tokens):
    return list(
        zip(*[tokens[:-1], tokens[1:]])
        )

In [17]:
collect_ngrams(tokens)[:10]

[('P', 'y'),
 ('y', 't'),
 ('t', 'h'),
 ('h', 'o'),
 ('o', 'n'),
 ('n', ' '),
 (' ', '('),
 ('(', 'p'),
 ('p', 'r'),
 ('r', 'o')]

But what if you want to collect trigrams for 4-grams? A list comprehension can help you iterate through the `range(n)` to extract the 1st, 2nd, ... N-th character.

In [18]:
def collect_ngrams(tokens, n=3):
    return list(
        zip(*[
                tokens[i:(-n+i)] for i in range(n)
            ])
        )

In [20]:
collect_ngrams(tokens, n=4)[:10]

[('P', 'y', 't', 'h'),
 ('y', 't', 'h', 'o'),
 ('t', 'h', 'o', 'n'),
 ('h', 'o', 'n', ' '),
 ('o', 'n', ' ', '('),
 ('n', ' ', '(', 'p'),
 (' ', '(', 'p', 'r'),
 ('(', 'p', 'r', 'o'),
 ('p', 'r', 'o', 'g'),
 ('r', 'o', 'g', 'r')]

And what if you want those n-grams to be put into a single string, instead of a tuple of strings?

In [25]:
def collect_ngrams(tokens, n=3, ngram_type=tuple):
    n_tuples = list(
        zip(*[
                tokens[i:(-n+i)] for i in range(n)
            ])
        )
    if isinstance(n_tuples[0], ngram_type):
        return n_tuples
    return [''.join(ngram) for ngram in n_tuples]        

In [26]:
collect_ngrams(tokens, n=4)[:10]

[('P', 'y', 't', 'h'),
 ('y', 't', 'h', 'o'),
 ('t', 'h', 'o', 'n'),
 ('h', 'o', 'n', ' '),
 ('o', 'n', ' ', '('),
 ('n', ' ', '(', 'p'),
 (' ', '(', 'p', 'r'),
 ('(', 'p', 'r', 'o'),
 ('p', 'r', 'o', 'g'),
 ('r', 'o', 'g', 'r')]

In [27]:
bigrams = collect_ngrams(tokens=tokens, n=2, ngram_type=str)
bigrams[:5]

['Py', 'yt', 'th', 'ho', 'on']

In [28]:
from collections import Counter
counts = Counter(bigrams)
for bg, count in list(counts.items())[:5]:
    print(bg, count)

Py 1926
yt 2446
th 5589
ho 2769
on 7236


In [32]:
sorted_counts = sorted([(count, ngram) for (ngram, count) in counts.items()])

In [33]:
sorted_counts[:10]

[(1, '\n"'),
 (1, "\n'"),
 (1, '\nJ'),
 (1, '\nK'),
 (1, '\nY'),
 (1, '\nl'),
 (1, '\nn'),
 (1, '\nq'),
 (1, '\nv'),
 (1, '\nw')]

In [34]:
sorted_counts[-10:]

[(4591, 'at'),
 (5071, 'er'),
 (5232, 'ti'),
 (5393, 'or'),
 (5502, ']('),
 (5589, 'th'),
 (5883, 'e '),
 (6230, 'te'),
 (6493, 'in'),
 (7236, 'on')]

Interesting that an e followed by a space is among the 4th most frequent 2-gram.

Probabilities (or _frequencies_) can be computed by dividing a count of a particular event (a particular n-gram) by the total number of possibile events, the total count of all the 2-grams in the text.

In [35]:
frequencies, cumulative_frequencies = [], [] 
for (bg, count) in counts.items():
   frequencies.append(count / len(bigrams))
frequencies[:6]

[0.00326602872273858,
 0.004147822562730305,
 0.009477588022526441,
 0.004695552197955934,
 0.01227050043496177,
 0.006913602857012042]

In [54]:
def compute_last_char_probabilities(counts):
    last_char_choices = dict()
    probabilities = dict()
    for ngram in counts:
        first_chars = ngram[:-1]
        last_char = ngram[-1]
        last_char_choices[first_chars] = last_char_choices.get(first_chars, [])
        last_char_choices[first_chars].append(last_char)
        probabilities[first_chars] = probabilities.get(first_chars, [])
        probabilities[first_chars].append(counts[ngram])
    return last_char_choices, probabilities

In [59]:
last_char_choices, probabilities = compute_last_char_probabilities(counts=Counter(bigrams))

In [56]:
sorted(zip(probabilities['e'], last_char_choices['e']))[-10:]

[(1295, 't'),
 (1328, 'l'),
 (1512, 'm'),
 (1674, 'c'),
 (1895, '_'),
 (3075, 's'),
 (3761, 'd'),
 (4213, 'n'),
 (5071, 'r'),
 (5883, ' ')]

You can see that a space (" ") is the most likely character to follow an "e", because a lot of words end in "e".

The `random.choices` function might be just what we need to chose from among the possible next letters.

In [47]:
import random
help(random.choices)

Help on method choices in module random:

choices(population, weights=None, *, cum_weights=None, k=1) method of random.Random instance
    Return a k sized list of population elements chosen with replacement.
    
    If the relative weights or cumulative weights are not specified,
    the selections are made with equal probability.



You can use the list of `last_char_choices` for the `population` argument in `random.choices()`. For the `weights` argument you can use the list of probabilities you computed for each of those character choices. Here's how you could make a random choice for the character to follow "e" in our 2-gram model.

In [48]:
random.choices(population=last_char_choices['e'], weights=probabilities['e'])

['n']

In [49]:
random.choices(population=last_char_choices['e'], weights=probabilities['e'])

['d']

In [50]:
random.choices(population=last_char_choices['e'], weights=probabilities['e'])

['t']

Well that's not so great. Chosing the most popular letter based on only one previous character doesn't work too well. So you probably want to increase n from 2 to 3 or 4, to get more interesting words.

In [58]:
word = ' '
c = ''
while c != ' ':
    c = random.choices(population=last_char_choices[word[-1]], weights=probabilities[word[-1]], k=1)[0]
    word += c
word

KeyError: ' '

In [74]:
trigrams = collect_ngrams(text, n=3, ngram_type=str)
last_char_choices, probabilities = compute_last_char_probabilities(counts=Counter(trigrams))
sorted(zip(probabilities[' P'], last_char_choices[' P']))[-10:]

[(11, 'i'),
 (12, 'D'),
 (13, '.'),
 (13, 'h'),
 (20, 'u'),
 (25, 'o'),
 (44, 'e'),
 (55, 'a'),
 (114, 'r'),
 (731, 'y')]

In [75]:
n = 3
word = ' P'
c = ''
while c != ' ':
    first_chars = word[-n+1:]
    c = random.choices(population=last_char_choices[first_chars], weights=probabilities[first_chars], k=1)[0]
    word += c
word

' Progrogra")***](/webash\n\n# '

That's starting to look a tiny bit more like actual words.

In [105]:
import string
def generate_word(last_char_choices, probabilities=None, start=None):
    n = len(next(iter(last_char_choices.keys()))) + 1
    word = start or random.choice(list(last_char_choices.keys()))
    while True:
        first_chars = word[-n+1:]
        if probabilities:
            c = random.choices(population=last_char_choices[first_chars], weights=probabilities[first_chars], k=1)[0]
        else:
            c = random.choices(population=last_char_choices[first_chars], k=1)[0]
        if c in string.punctuation + string.whitespace:
            break
        word += c
    return word

In [108]:
[generate_word(last_char_choices, probabilities) for i in range(10)]

['7975783',
 'NOP',
 '்தான்',
 'ไทยาการคอมพิวเตอร์',
 'Phivald',
 'C11',
 '+Abotente',
 'ენა',
 ')&olsouppreal',
 'ėka']

In [107]:
generate_word(last_char_choices, probabilities, start=' P')

' Pythonizary'

To get this working well, there's much more work to be done. You will need to reduce the vocabulary size so your algorithm doesn't have to generate rare character like those Arabic symbols. And you probably want to ignore capitalization. Another trick that works for chatbots is to keep track of a special "End of String" token (EOS)" so that you know when to start and stop generating text.